# 03 — Locked Held-Out Test Evaluation (`MultiCamCows2024` Days 19–20 + `NyalaData` Test) & Publication Figures
Runs ONCE on the locked test sets using the confidence rules and adapter weights frozen in Step 02. Generates exact H1–H4 verdicts, bootstrap 95% CIs, figures, and visual failure sheets.

In [ ]:
import json, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.metrics import roc_auc_score, roc_curve

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class MiewIDBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("efficientnetv2_rw_m", pretrained=False, num_classes=0)
        self.backbone.global_pool = GeM()
        self.bn = nn.BatchNorm1d(2152)
    def forward(self, x):
        return self.bn(self.backbone(x).view(x.shape[0], -1))

def load_miewid_msv3(device):
    sf_path = hf_hub_download("conservationxlabs/miewid-msv3", "model.safetensors")
    sd = load_file(sf_path)
    feat_sd = {k: v for k, v in sd.items() if k.startswith("backbone.") or k.startswith("bn.")}
    m = MiewIDBackbone()
    missing, unexpected = m.load_state_dict(feat_sd, strict=True)
    assert len(missing) == 0 and len(unexpected) == 0, f"Mismatch: {missing} | {unexpected}"
    print(f"Loaded official conservationxlabs/miewid-msv3 ({len(feat_sd)} tensors, GeM p={m.backbone.global_pool.p.item():.3f})")
    return m.to(device).eval()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)

split_cows_path = next(Path("/kaggle/input").rglob("splits_cows.csv"))
split_nyala_path = next(Path("/kaggle/input").rglob("splits_nyala.csv"))
val_sum_path = next(Path("/kaggle/input").rglob("val_summary.json"))
gal_npz_path = next(Path("/kaggle/input").rglob("gallery_embeddings.npz"))
adapter_pt_path = next(Path("/kaggle/input").rglob("miewid_adapter.pt"))

cows = pd.read_csv(split_cows_path)
nyala = pd.read_csv(split_nyala_path)
val_sum = json.loads(val_sum_path.read_text())
locked = val_sum["locked_rules_for_test"]
gal_npz = np.load(gal_npz_path, allow_pickle=True)
g_ids = gal_npz["g_ids"]
g_cams = gal_npz["g_cams"]
miew_gal = gal_npz["miew_gal"]
mega_gal = gal_npz["mega_gal"]
miew_adapt_gal = gal_npz["miew_adapt_gal"]

tk = cows[cows["role"] == "test_known"].copy().reset_index(drop=True)
tu = cows[cows["role"] == "test_unknown"].copy().reset_index(drop=True)
ny_test = nyala[nyala["split"] == "test"].copy().reset_index(drop=True)
print(f"Locked test: cows known={len(tk)}, cows unknown={len(tu)}, nyala test={len(ny_test)}")

def apply_degradation(img, mode):
    if mode == "clean": return img
    if mode == "low_light": return ImageEnhance.Brightness(img).enhance(0.35)
    if mode == "blur": return img.filter(ImageFilter.GaussianBlur(radius=3.0))
    if mode == "crop70":
        w, h = img.size
        nw, nh = int(w * 0.70), int(h * 0.70)
        l, t = (w - nw) // 2, (h - nh) // 2
        return img.crop((l, t, l + nw, t + nh))
    raise ValueError(mode)

class ImgDS(Dataset):
    def __init__(self, paths, tfm, mode="clean"):
        self.paths, self.tfm, self.mode = list(paths), tfm, mode
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            im = im.convert("RGB")
            if self.mode != "clean": im = apply_degradation(im, self.mode)
            return self.tfm(im)

@torch.no_grad()
def extract_embeddings(model, paths, tfm, batch_size=48, mode="clean"):
    dl = DataLoader(ImgDS(paths, tfm, mode=mode), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    embs = []
    for xb in dl:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            out = model(xb)
        if isinstance(out, (tuple, list)): out = out[0]
        elif hasattr(out, "last_hidden_state"): out = out.last_hidden_state.mean(dim=1)
        embs.append(F.normalize(out.float(), p=2, dim=-1).cpu().numpy())
    return np.concatenate(embs, axis=0)

miew_tfm = T.Compose([T.Resize((440, 440)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
mega_tfm = T.Compose([T.Resize((384, 384)), T.ToTensor(), T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

# Extract MiewID-msv3
miew = load_miewid_msv3(DEVICE)
miew_tk = extract_embeddings(miew, tk["abs_path"], miew_tfm)
miew_tu = extract_embeddings(miew, tu["abs_path"], miew_tfm)
miew_ny = extract_embeddings(miew, ny_test["abs_path"], miew_tfm)
miew_tk_deg = {m: extract_embeddings(miew, tk["abs_path"], miew_tfm, mode=m) for m in ["low_light", "blur", "crop70"]}
del miew; torch.cuda.empty_cache()

# Apply frozen adapter from step 02
class ResidualAdapter(nn.Module):
    def __init__(self, dim=2152, hidden=512, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, dim))
        self.alpha = nn.Parameter(torch.tensor(0.25))
    def forward(self, x): return F.normalize(x + self.alpha * self.net(x), p=2, dim=-1)

ckpt = torch.load(adapter_pt_path, map_location=DEVICE, weights_only=True)
adapter = ResidualAdapter(dim=ckpt["dim"]).to(DEVICE)
adapter.load_state_dict(ckpt["state_dict"])
adapter.eval()
with torch.no_grad():
    miew_ad_tk = adapter(torch.tensor(miew_tk, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    miew_ad_tu = adapter(torch.tensor(miew_tu, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    miew_ad_tk_deg = {m: adapter(torch.tensor(v, dtype=torch.float32, device=DEVICE)).cpu().numpy() for m, v in miew_tk_deg.items()}

# Extract MegaDescriptor-L-384
mega = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True).to(DEVICE).eval()
mega_tk = extract_embeddings(mega, tk["abs_path"], mega_tfm)
mega_tu = extract_embeddings(mega, tu["abs_path"], mega_tfm)
mega_ny = extract_embeddings(mega, ny_test["abs_path"], mega_tfm)
mega_tk_deg = {m: extract_embeddings(mega, tk["abs_path"], mega_tfm, mode=m) for m in ["low_light", "blur", "crop70"]}
del mega; torch.cuda.empty_cache()

def eval_retrieval(q_emb, q_ids, g_emb, g_ids, q_cams=None, g_cams=None, cam_mode="all", leave_one_out=False):
    sims = q_emb @ g_emb.T
    n_q = len(q_ids)
    r1 = np.zeros(n_q); r5 = np.zeros(n_q); aps = np.zeros(n_q)
    top1_sim = np.zeros(n_q); id_margin = np.zeros(n_q); pred_ids = []
    valid = np.ones(n_q, dtype=bool)
    g_ids_arr = np.asarray(g_ids)
    for i in range(n_q):
        row = sims[i].copy()
        if leave_one_out: row[i] = -1e9
        if cam_mode == "same" and q_cams is not None: mask = (g_cams == q_cams[i])
        elif cam_mode == "cross" and q_cams is not None: mask = (g_cams != q_cams[i])
        else: mask = np.ones(len(g_ids_arr), dtype=bool)
        if leave_one_out: mask[i] = False
        if not np.any(mask & (g_ids_arr == q_ids[i])):
            valid[i] = False; pred_ids.append(""); continue
        row[~mask] = -1e9
        order = np.argsort(-row)
        r_ids = g_ids_arr[order]; r_sims = row[order]
        pred_ids.append(str(r_ids[0]))
        r1[i] = float(r_ids[0] == q_ids[i])
        r5[i] = float(q_ids[i] in r_ids[:5])
        rel = np.where(r_ids == q_ids[i])[0]
        aps[i] = float(((np.arange(1, len(rel)+1)) / (rel + 1)).mean())
        top1_sim[i] = float(r_sims[0])
        other = r_sims[(r_ids != r_ids[0]) & (r_sims > -1e8)]
        id_margin[i] = top1_sim[i] - (float(other[0]) if len(other) else 0.0)
    return {
        "n_valid": int(valid.sum()), "r1": float(r1[valid].mean()), "r5": float(r5[valid].mean()),
        "mAP": float(aps[valid].mean()), "r1_arr": r1[valid], "top1_sim": top1_sim[valid],
        "id_margin": id_margin[valid], "valid_mask": valid, "pred_ids": np.array(pred_ids)[valid]
    }

def boot_ci(arr, n_boot=1000):
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(arr), size=(n_boot, len(arr)))
    m = arr[idx].mean(axis=1)
    return [round(float(np.percentile(m, 2.5)), 4), round(float(np.percentile(m, 97.5)), 4)]

def openset_eval(k_emb, u_emb, g_emb, g_ids):
    g_ids_arr = np.asarray(g_ids)
    def score(emb):
        sims = emb @ g_emb.T
        t1, mg = [], []
        for i in range(len(emb)):
            o = np.argsort(-sims[i]); ri = g_ids_arr[o]; rs = sims[i][o]
            t1.append(float(rs[0]))
            ot = rs[ri != ri[0]]
            mg.append(float(rs[0] - (ot[0] if len(ot) else 0.0)))
        return np.array(t1), np.array(mg)
    kt1, kmg = score(k_emb); ut1, umg = score(u_emb)
    y = np.concatenate([np.ones(len(kt1)), np.zeros(len(ut1))])
    out = {}
    for nm, s in [("top1_sim", np.concatenate([kt1, ut1])), ("id_margin", np.concatenate([kmg, umg]))]:
        auc = float(roc_auc_score(y, s))
        fpr, tpr, _ = roc_curve(1 - y, -s)
        tpr5 = float(tpr[np.searchsorted(fpr, 0.05, side="right") - 1]) if np.any(fpr <= 0.05) else 0.0
        out[nm] = {"auroc": round(auc, 4), "unk_tpr_at_5fpr": round(tpr5, 4)}
    return out, (kt1, ut1, kmg, umg)

tk_ids = tk["identity"].to_numpy(); tk_cams = tk["camera"].to_numpy()
ny_ids = ny_test["identity"].to_numpy(); ny_ori = ny_test["orientation"].to_numpy()

test_report = {"locked_rules_used": locked}
saved_miew = {}

for mname, qk, qu, ge, degs, ny_e in [
    ("miewid_msv3", miew_tk, miew_tu, miew_gal, miew_tk_deg, miew_ny),
    ("miewid_adapted", miew_ad_tk, miew_ad_tu, miew_adapt_gal, miew_ad_tk_deg, None),
    ("megadescriptor_l384", mega_tk, mega_tu, mega_gal, mega_tk_deg, mega_ny),
]:
    r_all = eval_retrieval(qk, tk_ids, ge, g_ids, tk_cams, g_cams, "all")
    r_same = eval_retrieval(qk, tk_ids, ge, g_ids, tk_cams, g_cams, "same")
    r_cross = eval_retrieval(qk, tk_ids, ge, g_ids, tk_cams, g_cams, "cross")
    os_res, os_raw = openset_eval(qk, qu, ge, g_ids)
    y_err = 1 - r_all["r1_arr"]
    err_aucs = {
        "top1_sim": round(float(roc_auc_score(y_err, -r_all["top1_sim"])), 4) if 0 < y_err.sum() < len(y_err) else None,
        "id_margin": round(float(roc_auc_score(y_err, -r_all["id_margin"])), 4) if 0 < y_err.sum() < len(y_err) else None,
    }
    # Risk-coverage at 10% and 20% abstention using locked closed-set score
    lscore = r_all[locked["locked_closed_set_score"]]
    order_conf = np.argsort(-lscore)
    r1_90 = round(float(r_all["r1_arr"][order_conf[:int(0.9*len(lscore))]].mean()), 4)
    r1_80 = round(float(r_all["r1_arr"][order_conf[:int(0.8*len(lscore))]].mean()), 4)

    deg_out = {}
    for dm, de in degs.items():
        dr = eval_retrieval(de, tk_ids, ge, g_ids)
        deg_out[dm] = {"r1": round(dr["r1"], 4), "r1_ci": boot_ci(dr["r1_arr"]), "mAP": round(dr["mAP"], 4), "drop_r1": round(r_all["r1"] - dr["r1"], 4)}

    entry = {
        "cross_day_test": {"n": r_all["n_valid"], "r1": round(r_all["r1"], 4), "r1_ci": boot_ci(r_all["r1_arr"]), "r5": round(r_all["r5"], 4), "mAP": round(r_all["mAP"], 4)},
        "same_cam_test": {"n": r_same["n_valid"], "r1": round(r_same["r1"], 4), "r1_ci": boot_ci(r_same["r1_arr"]), "mAP": round(r_same["mAP"], 4)},
        "cross_cam_test": {"n": r_cross["n_valid"], "r1": round(r_cross["r1"], 4), "r1_ci": boot_ci(r_cross["r1_arr"]), "mAP": round(r_cross["mAP"], 4)},
        "cam_drop_r1": round(r_same["r1"] - r_cross["r1"], 4),
        "degradations_test": deg_out,
        "closed_set_error_auroc": err_aucs,
        "selective_r1_at_90pct_cov": r1_90,
        "selective_r1_at_80pct_cov": r1_80,
        "openset_unknown_test": os_res,
    }
    if ny_e is not None:
        ny_same = eval_retrieval(ny_e, ny_ids, ny_e, ny_ids, ny_ori, ny_ori, "same", leave_one_out=True)
        ny_opp = eval_retrieval(ny_e, ny_ids, ny_e, ny_ids, ny_ori, ny_ori, "cross", leave_one_out=True)
        entry["nyala_same_side_test"] = {"n": ny_same["n_valid"], "r1": round(ny_same["r1"], 4), "r1_ci": boot_ci(ny_same["r1_arr"]), "mAP": round(ny_same["mAP"], 4)}
        entry["nyala_opposite_side_test"] = {"n": ny_opp["n_valid"], "r1": round(ny_opp["r1"], 4), "r1_ci": boot_ci(ny_opp["r1_arr"]), "mAP": round(ny_opp["mAP"], 4)}
        entry["nyala_viewpoint_drop_r1"] = round(ny_same["r1"] - ny_opp["r1"], 4)
    test_report[mname] = entry
    if mname == "miewid_msv3":
        saved_miew = {"r_all": r_all, "r_cross": r_cross, "os_raw": os_raw}

# Save summary JSON
(OUT / "test_summary.json").write_text(json.dumps(test_report, indent=2))
print(json.dumps(test_report, indent=2))

# --- FIGURES ---
plt.rcParams.update({"font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False})

# Fig 1: Imaging factors comparison (MiewID vs Adapted vs MegaDescriptor)
fig, ax = plt.subplots(figsize=(9.6, 4.6), dpi=160)
labels = ["Random split\n(val pool)", "Unseen days\n(test)", "Same cam\n(test)", "Cross cam\n(test)", "Nyala\nsame side", "Nyala\nopposite side"]
miew_vals = [
    val_sum["miewid_msv3"]["random_split"]["r1"],
    test_report["miewid_msv3"]["cross_day_test"]["r1"],
    test_report["miewid_msv3"]["same_cam_test"]["r1"],
    test_report["miewid_msv3"]["cross_cam_test"]["r1"],
    test_report["miewid_msv3"]["nyala_same_side_test"]["r1"],
    test_report["miewid_msv3"]["nyala_opposite_side_test"]["r1"],
]
adapt_vals = [
    np.nan,
    test_report["miewid_adapted"]["cross_day_test"]["r1"],
    test_report["miewid_adapted"]["same_cam_test"]["r1"],
    test_report["miewid_adapted"]["cross_cam_test"]["r1"],
    np.nan,
    np.nan,
]
mega_vals = [
    val_sum["megadescriptor_l384"]["random_split"]["r1"],
    test_report["megadescriptor_l384"]["cross_day_test"]["r1"],
    test_report["megadescriptor_l384"]["same_cam_test"]["r1"],
    test_report["megadescriptor_l384"]["cross_cam_test"]["r1"],
    test_report["megadescriptor_l384"]["nyala_same_side_test"]["r1"],
    test_report["megadescriptor_l384"]["nyala_opposite_side_test"]["r1"],
]
x = np.arange(len(labels)); w = 0.26
b1 = ax.bar(x - w, [v*100 for v in miew_vals], w, label="MiewID-msv3 (zero-shot)", color="#17365D")
b2 = ax.bar(x, [v*100 if not np.isnan(v) else 0 for v in adapt_vals], w, label="MiewID + Gallery Adapter (cows)", color="#2E7D32")
b3 = ax.bar(x + w, [v*100 for v in mega_vals], w, label="MegaDescriptor-L-384*", color="#5B9BD5")
for bars, vals in [(b1, miew_vals), (b2, adapt_vals), (b3, mega_vals)]:
    for bar, v in zip(bars, vals):
        if np.isnan(v): continue
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1.2, f"{h:.1f}%", ha="center", va="bottom", fontsize=7.8)
ax.set_ylim(0, 114); ax.set_ylabel("Rank-1 Accuracy (%)")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Effect of Time, Camera, and Viewpoint Factors on Bovid Re-ID")
ax.legend(frameon=False, loc="upper right", fontsize=8.8)
fig.text(0.01, 0.005, "*MegaDescriptor-L-384 pre-training overlaps NyalaData (2023) but not MultiCamCows2024.", fontsize=7.8, color="#555555")
fig.tight_layout(rect=[0, 0.03, 1, 1]); fig.savefig(OUT / "fig1_imaging_factors.png", dpi=160); plt.close(fig)

# Fig 2: Field degradation impact on locked test set
fig, ax = plt.subplots(figsize=(8.2, 4.5), dpi=160)
dlabels = ["Clean", "Low light\n(0.35x)", "Blur\n(sigma=3)", "Tight crop\n(70%)"]
modes = ["clean", "low_light", "blur", "crop70"]
pretty = {"miewid_msv3": "MiewID-msv3 (zero-shot)", "miewid_adapted": "MiewID + Gallery Adapter", "megadescriptor_l384": "MegaDescriptor-L-384"}
offsets = {
    "megadescriptor_l384": [(0.0, 4.2), (0.0, 4.2), (0.0, 4.2), (0.0, -5.8)],
    "miewid_msv3":         [(-0.20, -1.2), (-0.21, -0.5), (0.0, -5.8), (-0.16, 3.8)],
    "miewid_adapted":      [(0.0, -6.2), (0.0, -6.2), (0.0, 4.2), (0.19, 1.0)],
}
for nm, col, mk in [("miewid_msv3", "#17365D", "o"), ("miewid_adapted", "#2E7D32", "s"), ("megadescriptor_l384", "#5B9BD5", "^")]:
    ys = [test_report[nm]["cross_day_test"]["r1"]*100] + [test_report[nm]["degradations_test"][m]["r1"]*100 for m in modes[1:]]
    ax.plot(range(len(dlabels)), ys, marker=mk, lw=2.2, ms=7, label=pretty[nm], color=col)
    for xi, yv in enumerate(ys):
        dx, dy = offsets[nm][xi]
        ax.text(xi + dx, yv + dy, f"{yv:.1f}%", ha="center", va="center", fontsize=8.4, fontweight="medium", color=col)
ax.set_xticks(range(len(dlabels))); ax.set_xticklabels(dlabels)
ax.set_xlim(-0.42, 3.42)
ax.set_ylim(0, 106); ax.set_ylabel("Rank-1 Accuracy (%) — Locked Test Days")
ax.set_title("Sensitivity to Handheld Field Degradations (MultiCamCows2024 Test)")
ax.grid(axis="y", alpha=0.3); ax.legend(frameon=False, loc="lower left")
fig.tight_layout(); fig.savefig(OUT / "fig2_degradations.png", dpi=160); plt.close(fig)

# Fig 3: Open-set score distributions (Known vs Unknown cows, MiewID-msv3)
kt1, ut1, kmg, umg = saved_miew["os_raw"]
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.9), dpi=160)
for ax, ks, us, title in [
    (axes[0], kt1, ut1, f"Top-1 Cosine Similarity (AUROC={test_report['miewid_msv3']['openset_unknown_test']['top1_sim']['auroc']:.3f})"),
    (axes[1], kmg, umg, f"Identity Margin Top1-Top2 (AUROC={test_report['miewid_msv3']['openset_unknown_test']['id_margin']['auroc']:.3f})"),
]:
    ax.hist(ks, bins=25, alpha=0.65, density=True, label="Known cows (70 IDs)", color="#17365D")
    ax.hist(us, bins=25, alpha=0.65, density=True, label="Unseen cows (10 IDs)", color="#C0392B")
    ax.set_ylabel("Density")
    ax.set_title(title, fontsize=9.8); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT / "fig3_openset_distributions.png", dpi=160); plt.close(fig)

# Fig 4: Real failure-case contact sheet (up to 6 worst MiewID cross-day errors)
r_all = saved_miew["r_all"]
tk_valid = tk[r_all["valid_mask"]].reset_index(drop=True)
err_idx = np.where(r_all["r1_arr"] == 0)[0]
if len(err_idx) > 0:
    # Sort by highest confidence errors (most dangerous silent failures!)
    err_sorted = err_idx[np.argsort(-r_all["top1_sim"][err_idx])][:6]
    fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.0), dpi=150)
    for ax, ei in zip(axes.flat, err_sorted):
        im = Image.open(tk_valid.loc[ei, "abs_path"]).convert("RGB")
        ax.imshow(im); ax.axis("off")
        ax.set_title(f"True: {tk_valid.loc[ei,'identity'].split('_')[-1]} | Pred: {r_all['pred_ids'][ei].split('_')[-1]}\nCam {tk_valid.loc[ei,'camera']} | sim={r_all['top1_sim'][ei]:.2f} | margin={r_all['id_margin'][ei]:.2f}", fontsize=8.4)
    for ax in axes.flat[len(err_sorted):]: ax.axis("off")
    fig.suptitle("Highest-Confidence Misidentifications (MiewID-msv3, Locked Test Days)", fontsize=10.5)
    fig.tight_layout(); fig.savefig(OUT / "fig4_failure_cases.png", dpi=150); plt.close(fig)
print("All figures saved cleanly.")
